In [1]:
import os
from pathlib import Path

# jupyter sets the kernel cwd to this notebook's directory; hop to the repo
# root so imports and relative paths (data/, splits/, backbone/) resolve
if Path.cwd().name == "ensemble":
  os.chdir(Path.cwd().parent)

from dataclasses import asdict

import torch as t
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models

from torchinfo import summary

from aim import Run

from ensemble.train import train, TrainConfig, TrainJob
from ensemble.eval import evaluate, plot_loss, plot_metrics

from backbone.resnet20 import ResNet20 

/home/john/Desktop/Grad Code/chatty-networks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def mem(label):                                                                            
  alloc = t.cuda.memory_allocated() / 1e9                                            
  res   = t.cuda.memory_reserved()  / 1e9                                            
  peak  = t.cuda.max_memory_allocated() / 1e9                                        
  print(f"{label:20s}  allocated={alloc:.3f} GB  reserved={res:.3f} GB  peak={peak:.3f} GB")                                                                                         
                                                                                              
t.cuda.reset_peak_memory_stats() 

# Setup

### Configs

In [ ]:
train_cfg = TrainConfig(
  epochs=2,
  lr=1e-3,
  weight_decay=1e-4,
  seed=0,
  device="cuda" if t.cuda.is_available() else "cpu",
)

t.manual_seed(train_cfg.seed)
t.cuda.manual_seed_all(train_cfg.seed)
print(f"device: {train_cfg.device}")

### Load ckpts + models

In [4]:
ckpt_paths = ["backbone/checkpoints/stratified_backbone_seed42_epoch200.pt", "backbone/checkpoints/stratified_backbone_seed137_epoch200.pt"]

n_models = 2

assert n_models == len(ckpt_paths)

ckpts = [t.load(ckpt_paths[i], map_location=train_cfg.device, weights_only=True) for i in range(n_models)]

FileNotFoundError: [Errno 2] No such file or directory: 'backbone/checkpoints/stratified_backbone_seed42_epoch200.pt'

In [ ]:
models = [ResNet20() for _ in range(n_models)]

for i in range(n_models):
  models[i].load_state_dict(ckpts[i]["state_dict"])
  models[i].to(train_cfg.device)

### Load data

In [ ]:
batch_size = 128
image_shape = (1, 3, 32, 32)
num_classes = 100

_CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
_CIFAR100_STD = (0.2675, 0.2565, 0.2761)

train_tf = transforms.Compose([
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(),
  transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
])
eval_tf = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
])

# canonical 3-way split of the 50k train set (see splits/generate.py);
# the ensemble trains on ensemble_indices, which the backbones never saw
split_file = "splits/three_way_seed0.pt"
split = t.load(split_file, weights_only=True)

train_dataset = datasets.CIFAR100("./data", train=True, transform=train_tf, download=True)
val_dataset = datasets.CIFAR100("./data", train=True, transform=eval_tf, download=True)
test_dataset = datasets.CIFAR100("./data", train=False, transform=eval_tf, download=True)

ensemble_train = Subset(train_dataset, split["ensemble_indices"])
ensemble_val = Subset(val_dataset, split["val_indices"])

ensemble_train_loader = DataLoader(ensemble_train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(ensemble_val, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

len(ensemble_train), len(ensemble_val), len(test_dataset)

### Setup orchestrator

In [ ]:
from ensemble.communicative.driver import Driver
from ensemble.communicative.bus import QKVEncoder, Decoder
from ensemble.communicative.orchestrator import Specialist, Orchestrator

In [ ]:
stats = summary(models[0], input_size=image_shape)

stats

In [ ]:
def get_info(m: nn.Module):
  for info in stats.summary_list:
    if info.module is m:
      return info

In [ ]:
early_name = "layer1.2"
late_name = "layer3.0"

# assume homogeny
drivers = [Driver(models[i], early_name, late_name) for i in range(n_models)]

# early <- decoder: want input shape. output shape for encoder
early_shape = get_info(drivers[0].early).input_size
late_shape = get_info(drivers[0].late).output_size

# shape: [B, C, H, W]

early_shape, late_shape

In [ ]:
from einops.layers.torch import Reduce, Rearrange

key_dim = 16    # signature/query dim (TarMAC used 16)
value_dim = 64  # message content dim, consumed by the decoder

# TODO(ali): try something else as well
expanders = [Rearrange("b c -> b c 1 1") for _ in range(2)]

# try mean pooling. TODO(ali): try flattening
reducers = [Reduce("b c h w -> b c", "mean") for _ in range(2)]


adapters = [
  {
    "decoder": Decoder(value_dim, early_shape[1], expanders[i]),
    "encoder": QKVEncoder(late_shape[1], key_dim, value_dim, reducers[i])
  }
  for i in range(2)
]

In [ ]:
specialists = [Specialist(drivers[i], **adapters[i]) for i in range(n_models)]

# the trained target; the TarMAC bus is parameterless and constructed internally
orchestrator = Orchestrator(specialists, key_dim, value_dim, num_classes)

## optimizer + loss

In [ ]:
optimizer = optim.AdamW(orchestrator.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
# the orchestrator returns log of prob-averaged softmaxes, so NLLLoss, not CE
criterion = nn.NLLLoss()

## aim run

Hparams logged for filtering/comparing runs later.

In [ ]:
run = Run(experiment="mvp")
run["hparams"] = {
  **asdict(train_cfg),
  "batch_size": batch_size,
  "image_shape": image_shape,
  "num_classes": num_classes,
  "key_dim": key_dim,
  "value_dim": value_dim,
  "split_file": split_file,
  "model": "orchestrator-tarmac",
  "dataset": "cifar100",
  "backbones": ["ResNet20", "ResNet20"]
}
print(f"aim run hash: {run.hash}")

## train

In [ ]:
job = TrainJob(
  model=orchestrator,
  loader=ensemble_train_loader,
  optimizer=optimizer,
  criterion=criterion,
  config=train_cfg,
  run=run,
  val_loader=val_loader,
)

losses = train(job, k_rounds=1)

## loss curve

In [ ]:
plot_loss(losses, smooth=50)

## evaluate on test

In [ ]:
val_results = evaluate(orchestrator, val_loader, device=train_cfg.device, criterion=criterion)
print("val:", val_results)
for k, v in val_results.items():
  run.track(v, name=f"val_{k}")

results = evaluate(orchestrator, test_loader, device=train_cfg.device, criterion=criterion)
print("test:", results)
for k, v in results.items():
  run.track(v, name=f"test_{k}")

## metrics

In [ ]:
plot_metrics(results)